# Validación de Simulador de Series de Tiempo Funcionales FAR(1)

**Objetivo:** Validar la implementación del simulador FAR(1) que genera datos funcionales con estructura de dependencia temporal, tendencia determinística y ruido.

**Conceptos clave:**
- Dato funcional: Cada $X_t$ es una función $X_t(s)$ evaluada en $s \in [0,1]$.
- FAR(1): Proceso autorregresivo funcional de orden 1.
- Operador integral $\Psi$: Captura la dependencia temporal entre curvas.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, normaltest
from scipy import stats
import sys
sys.path.append('..')  # Ajusta según tu estructura de carpetas

# Importar las clases del simulador
from simulation_pipeline_linear import (
    FunctionalDomain,
    FARSimulator,
    gaussian_integral_kernel,
    build_integral_matrix
)

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['figure.dpi'] = 100

print("✓ Módulos importados correctamente")

## 1. Dominio Funcional

El dominio funcional es la malla de puntos donde se evalúan las curvas. Una malla más fina da mejor aproximación a la función continua, pero aumenta el costo computacional.

In [ ]:
domains = {
    'Grueso (20 pts)': FunctionalDomain.regular(s_min=0, s_max=1, n_points=20),
    'Medio (50 pts)': FunctionalDomain.regular(s_min=0, s_max=1, n_points=50),
    'Fino (100 pts)': FunctionalDomain.regular(s_min=0, s_max=1, n_points=100),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 3))
for ax, (name, domain) in zip(axes, domains.items()):
    ax.scatter(domain.grid, np.zeros_like(domain.grid), 
              s=50, alpha=0.6, c='steelblue')
    ax.set_title(f'{name}\n{domain.n_points} puntos de evaluación')
    ax.set_xlabel('s (dominio)')
    ax.set_ylim(-0.1, 0.1)
    ax.grid(True, alpha=0.3)

plt.suptitle('Diferentes resoluciones del dominio funcional [0,1]', 
             fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

## 2. Operador Integral Ψ

La matriz $\Psi$ discretiza el operador integral que define la dependencia temporal:

$$Y_t(s) = \int \Psi(s,u) Y_{t-1}(u) du + \varepsilon_t(s)$$

El kernel gaussiano implica dependencia local: el valor en $s$ depende más de valores cercanos en $u$.

**Estabilidad:** Para que el proceso sea estacionario, el valor propio máximo en módulo debe ser < 1.

In [ ]:
bandwidths = [0.02, 0.05, 0.1]
decay = 0.7
domain = FunctionalDomain.regular(n_points=50)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, bw in zip(axes, bandwidths):
    Psi = build_integral_matrix(domain, gaussian_integral_kernel, 
                               bandwidth=bw, decay=decay)
    im = ax.imshow(Psi, aspect='auto', cmap='RdBu_r',
                  extent=[0, 1, 1, 0])
    ax.set_title(f'Ψ con bandwidth={bw}\n(decay={decay})')
    ax.set_xlabel('u (entrada)')
    ax.set_ylabel('s (salida)')
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle('Operador integral Ψ(s,u) para diferentes anchos de banda', 
             fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

# Análisis de estabilidad
print("\n--- Análisis de Estabilidad del Operador ---")
for bw in bandwidths:
    Psi = build_integral_matrix(domain, gaussian_integral_kernel, 
                               bandwidth=bw, decay=decay)
    eigenvalues = np.linalg.eigvals(Psi)
    max_abs_eigen = np.max(np.abs(eigenvalues))
    print(f"Bandwidth={bw:.3f}: |λ|_max = {max_abs_eigen:.4f} ", end="")
    if max_abs_eigen < 1:
        print("✓ ESTABLE")
    else:
        print("⚠ INESTABLE")

## 3. Simulación FAR(1) con diferentes tendencias

Generamos 50 curvas con cuatro tendencias determinísticas. Se observa cómo la tendencia se suma al proceso estocástico base.

In [ ]:
domain = FunctionalDomain.regular(n_points=100)
Psi = build_integral_matrix(domain, gaussian_integral_kernel, 
                           bandwidth=0.05, decay=0.6)

trends = ['zero', 'linear', 'sinusoidal', 'bump']
simulators = {}

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
for ax, trend_name in zip(axes.flat, trends):
    sim = FARSimulator(
        domain=domain, n_curves=50, Psi=Psi,
        trend=trend_name, noise_std=0.3, noise_type='smooth',
        burn_in=100, random_state=42
    )
    X = sim.simulate()
    simulators[trend_name] = sim
    
    for t in range(30):
        ax.plot(domain.grid, X[t], alpha=0.3, linewidth=0.8)
    ax.set_title(f'Tendencia: {trend_name}')
    ax.set_xlabel('s')
    ax.set_ylabel('X_t(s)')
    ax.grid(True, alpha=0.3)

plt.suptitle('Procesos FAR(1) con diferentes tendencias determinísticas', 
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 4. Dependencia temporal: FAR(1) vs sin memoria

Comparamos el proceso FAR(1) (con memoria) contra un proceso donde $\Psi=0$ (curvas independientes). La diferencia se aprecia en la autocorrelación.

In [ ]:
sim_far = FARSimulator(
    domain=domain, n_curves=100, Psi=Psi,
    trend='sinusoidal', noise_std=0.3, noise_type='smooth',
    burn_in=100, random_state=42
)
X_far = sim_far.simulate()

Psi_zero = np.zeros_like(Psi)
sim_ind = FARSimulator(
    domain=domain, n_curves=100, Psi=Psi_zero,
    trend='sinusoidal', noise_std=0.3, noise_type='smooth',
    burn_in=0, random_state=42
)
X_ind = sim_ind.simulate()

fig, axes = plt.subplots(2, 2, figsize=(15, 8))

# Heatmaps
for ax, X, title in zip(axes[0], [X_far, X_ind], 
                         ['FAR(1) con memoria', 'Curvas Independientes']):
    im = ax.imshow(X, aspect='auto', cmap='RdBu_r',
                  extent=[0, 1, 100, 0], vmin=-3, vmax=3)
    ax.set_title(f'Heatmap: {title}')
    ax.set_xlabel('s')
    ax.set_ylabel('t')
    plt.colorbar(im, ax=ax)

# Autocorrelación en s=0.5
s_point = 0.5
idx = np.argmin(np.abs(domain.grid - s_point))
for ax, X, title in zip(axes[1], [X_far, X_ind],
                         ['FAR(1)', 'Independiente']):
    series_at_s = X[:, idx]
    corr_lag1 = pearsonr(series_at_s[:-1], series_at_s[1:])[0]
    ax.plot(series_at_s, linewidth=0.8, alpha=0.7)
    ax.set_title(f'{title}: X_t(s={s_point:.1f})\nCorr(t, t-1) = {corr_lag1:.3f}')
    ax.set_xlabel('t')
    ax.set_ylabel(f'X_t({s_point:.1f})')
    ax.grid(True, alpha=0.3)

plt.suptitle('Comparación: Proceso con memoria vs Sin memoria', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Validación estadística

Verificamos:
- Media funcional ≈ 0 (con tendencia cero)
- Estructura de autocorrelación temporal
- Normalidad marginal
- Matriz de covarianza

In [ ]:
sim_valid = FARSimulator(
    domain=domain, n_curves=500, Psi=Psi,
    trend='zero', noise_std=0.5, noise_type='smooth',
    burn_in=200, random_state=123
)
X_valid = sim_valid.simulate()

fig, axes = plt.subplots(2, 3, figsize=(16, 8))

# Media funcional
mean_curve = X_valid.mean(axis=0)
std_curve = X_valid.std(axis=0)
axes[0,0].plot(domain.grid, mean_curve, 'b-', linewidth=2, label='Media empírica')
axes[0,0].fill_between(domain.grid, 
                       mean_curve - 2*std_curve/np.sqrt(500),
                       mean_curve + 2*std_curve/np.sqrt(500),
                       alpha=0.3, label='IC 95%')
axes[0,0].axhline(y=0, color='r', linestyle='--', alpha=0.5)
axes[0,0].set_title('Media funcional (debería ser 0)')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# Varianza funcional
axes[0,1].plot(domain.grid, std_curve**2, 'b-', linewidth=2)
axes[0,1].set_title('Varianza funcional Var[X_t(s)]')
axes[0,1].set_xlabel('s')
axes[0,1].grid(True, alpha=0.3)

# Autocorrelación temporal en varios puntos s
for s_val in [0.2, 0.5, 0.8]:
    idx = np.argmin(np.abs(domain.grid - s_val))
    autocorr = [1.0]
    series = X_valid[:, idx]
    for lag in range(1, 20):
        corr = pearsonr(series[:-lag], series[lag:])[0]
        autocorr.append(corr)
    axes[0,2].plot(range(len(autocorr)), autocorr, 
                  marker='o', markersize=3, label=f's={s_val}')
axes[0,2].set_title('Autocorrelación temporal en s fijo')
axes[0,2].set_xlabel('Lag')
axes[0,2].set_ylabel('ACF')
axes[0,2].legend()
axes[0,2].grid(True, alpha=0.3)

# Distribución marginal en s=0.5
s_mid = 0.5
idx_mid = np.argmin(np.abs(domain.grid - s_mid))
axes[1,0].hist(X_valid[:, idx_mid], bins=30, density=True, 
              alpha=0.7, color='steelblue')
x_range = np.linspace(-3, 3, 100)
axes[1,0].plot(x_range, stats.norm.pdf(x_range, 0, X_valid[:, idx_mid].std()),
              'r-', linewidth=2, label='Normal ajustada')
axes[1,0].set_title(f'Distribución marginal en s={s_mid}')
axes[1,0].legend()

# Covarianza cruzada
s1, s2 = 0.3, 0.7
idx1 = np.argmin(np.abs(domain.grid - s1))
idx2 = np.argmin(np.abs(domain.grid - s2))
axes[1,1].scatter(X_valid[:, idx1], X_valid[:, idx2], 
                 alpha=0.3, s=10)
axes[1,1].set_xlabel(f'X_t(s={s1})')
axes[1,1].set_ylabel(f'X_t(s={s2})')
corr_cross = pearsonr(X_valid[:, idx1], X_valid[:, idx2])[0]
axes[1,1].set_title(f'Correlación cruzada: ρ={corr_cross:.3f}')

# Eigenvalores de la matriz de covarianza
Cov_emp = np.cov(X_valid.T)
eigenvals = np.linalg.eigvalsh(Cov_emp)
eigenvals = eigenvals[::-1]
axes[1,2].plot(range(1, 21), eigenvals[:20]/eigenvals.sum()*100, 'bo-')
axes[1,2].set_title('% Varianza explicada (20 primeros)')
axes[1,2].set_xlabel('Componente')
axes[1,2].grid(True, alpha=0.3)

plt.suptitle('Validación del Proceso FAR(1) Generado', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Test de normalidad
_, p_value = normaltest(X_valid[:, idx_mid])
print(f"Test de normalidad en s=0.5: p-value = {p_value:.4f}")
print(f"Media global: {mean_curve.mean():.6f}")
print(f"Varianza media: {std_curve.var():.4f}")

## 6. Análisis de sensibilidad

Exploramos cómo afectan `noise_std`, `bandwidth` y `burn_in`.

In [ ]:
# Efecto de la intensidad del ruido
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, noise in zip(axes, [0.1, 0.5, 1.0]):
    sim_noise = FARSimulator(
        domain=domain, n_curves=20, Psi=Psi,
        trend='sinusoidal', noise_std=noise, 
        noise_type='smooth', burn_in=100, random_state=42
    )
    X_noise = sim_noise.simulate()
    for t in range(20):
        ax.plot(domain.grid, X_noise[t], alpha=0.5, linewidth=0.8)
    ax.set_title(f'noise_std = {noise}')
    ax.set_ylim(-4, 4)
    ax.set_xlabel('s')
    ax.grid(True, alpha=0.3)
plt.suptitle('Impacto de la intensidad del ruido', fontsize=14)
plt.tight_layout()
plt.show()

# Importancia del burn-in
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, burn in zip(axes, [0, 20, 200]):
    sim_burn = FARSimulator(
        domain=domain, n_curves=30, Psi=Psi,
        trend='zero', noise_std=0.5, noise_type='smooth',
        burn_in=burn, random_state=42
    )
    X_burn = sim_burn.simulate()
    im = ax.imshow(X_burn, aspect='auto', cmap='RdBu_r',
                  extent=[0, 1, 30, 0])
    ax.set_title(f'burn_in = {burn}')
    ax.set_xlabel('s')
    ax.set_ylabel('t')
    plt.colorbar(im, ax=ax, shrink=0.8)
plt.suptitle('Efecto del burn-in', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Caso de uso: detección de anomalías funcionales

Insertamos curvas anómalas (shift, amplitud, forma) para probar métodos de detección.

In [ ]:
sim_normal = FARSimulator(
    domain=domain, n_curves=100, Psi=Psi,
    trend='zero', noise_std=0.3, noise_type='smooth',
    burn_in=100, random_state=42
)
X_normal = sim_normal.simulate()

X_with_anomalies = X_normal.copy()
anomaly_positions = [20, 50, 80]
anomaly_types = ['shift', 'amplitude', 'shape']

X_with_anomalies[20] += 3.0                      # desplazamiento vertical
X_with_anomalies[50] = X_normal[50] * 2.5       # mayor amplitud
X_with_anomalies[80] = np.sin(4 * np.pi * domain.grid) * 2  # forma diferente

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

im = axes[0].imshow(X_with_anomalies, aspect='auto', cmap='RdBu_r',
                   extent=[0, 1, 100, 0], vmin=-3, vmax=3)
for pos, atype in zip(anomaly_positions, anomaly_types):
    axes[0].axhline(y=pos, color='yellow', linestyle='--', 
                   linewidth=1.5, alpha=0.8)
    axes[0].text(1.02, pos/100, f'{atype}', 
                transform=axes[0].transAxes, color='red', fontweight='bold')
axes[0].set_title('Heatmap con anomalías')
axes[0].set_xlabel('s')
axes[0].set_ylabel('t')
plt.colorbar(im, ax=axes[0])

curves_to_plot = [19, 20, 21, 49, 50, 51, 79, 80, 81]
for idx, t in enumerate(curves_to_plot):
    color = 'red' if t in anomaly_positions else 'steelblue'
    alpha = 1.0 if t in anomaly_positions else 0.3
    linewidth = 2.5 if t in anomaly_positions else 1.0
    label = 'Anomalía' if t in anomaly_positions else 'Normal'
    axes[1].plot(domain.grid, X_with_anomalies[t], 
                color=color, alpha=alpha, linewidth=linewidth,
                label=label if idx < 3 else '')

handles, labels = axes[1].get_legend_handles_labels()
unique = dict(zip(labels, handles))
axes[1].legend(unique.values(), unique.keys())
axes[1].set_title('Curvas normales vs anómalas')
axes[1].set_xlabel('s')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Detección de Anomalías en Series de Tiempo Funcionales', fontsize=14)
plt.tight_layout()
plt.show()

## Resumen

✅ **Verificado:**
- Dominio funcional correcto.
- Operador Ψ bien construido, control de estabilidad.
- Dependencia temporal FAR(1) visible en autocorrelación.
- Tendencias agregadas correctamente.
- Ruido blanco/suave controlado.
- Burn-in estabiliza el proceso.

🎯 **Próximos pasos:** usar estos datos para ajustar modelos, predecir o detectar anomalías.